In [1]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path

LAST_NAME = "Papa"
STUDENT_ID = "TUPM-26-1927"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

RECORDS_TO_PROCESS = 1000 + ID_SUM
TARGET_CLASS = (SEED_DIGIT % 4) + 2

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Records to process:", RECORDS_TO_PROCESS)
print("Target status class:", TARGET_CLASS, f"({TARGET_CLASS}xx)")
print("Workspace:", Path(".").resolve())

Seed: 7
ID Sum: 27
Records to process: 1027
Target status class: 5 (5xx)
Workspace: C:\Users\Cyrus\OneDrive\Desktop\Lab6_File_Handling\Lab6_File_Handling_Papa\Lab6_File_Handling_Additional_Exercises


In [2]:
# [CELL 2] TEST THE PARSER
dataset_path = DATA_DIR / "weblog.csv"
print("Exists:", dataset_path.exists())

def parse_line(line):
    ip, rest = line.strip().split(",", 1)
    time_stamp, rest = rest.split(",", 1)
    url, status = rest.rsplit(",", 1)
    return ip, time_stamp.lstrip("["), url, status

with open(dataset_path, "r", encoding="utf-8") as file:
    header = file.readline().strip()
    print("Header:", header)
    for _ in range(3):
        print(parse_line(file.readline()))

Exists: True
Header: IP,Time,URL,Staus
('10.128.2.1', '29/Nov/2017:06:58:55', 'GET /login.php HTTP/1.1', '200')
('10.128.2.1', '29/Nov/2017:06:59:02', 'POST /process.php HTTP/1.1', '302')
('10.128.2.1', '29/Nov/2017:06:59:03', 'GET /home.php HTTP/1.1', '200')


In [3]:
# [CELL 3] ANALYZE THE LOG
records_processed = 0
invalid_records = 0
status_counts = {}
url_counts = {}
target_rows = []
target_prefix = str(TARGET_CLASS)

with open(dataset_path, "r", encoding="utf-8") as file:
    next(file)  # skip the header

    for line in file:
        if records_processed >= RECORDS_TO_PROCESS:
            break
        records_processed += 1

        try:
            ip, time_stamp, url, status = parse_line(line)
            path = url.split()[1]
        except (ValueError, IndexError):
            invalid_records += 1
            continue

        if len(status) != 3 or not status.isdigit():
            invalid_records += 1
            continue

        status_counts[status] = status_counts.get(status, 0) + 1

        if status.startswith(target_prefix):
            target_rows.append((ip, time_stamp, path, status))
            url_counts[path] = url_counts.get(path, 0) + 1

top_urls = sorted(url_counts.items(), key=lambda item: item[1], reverse=True)[:5]

print("Records processed:", records_processed)
print("Invalid records skipped:", invalid_records)
print("Target status class:", f"{TARGET_CLASS}xx")
print("Records in target class:", len(target_rows))
print("Count per status code:", dict(sorted(status_counts.items())))
print("Five most requested URLs in target class:")
if top_urls:
    for path, count in top_urls:
        print(f"  {path}: {count}")
else:
    print("  (no records in the target class)")

Records processed: 1027
Invalid records skipped: 0
Target status class: 5xx
Records in target class: 0
Count per status code: {'200': 849, '206': 8, '302': 144, '304': 21, '404': 5}
Five most requested URLs in target class:
  (no records in the target class)


In [4]:
# [CELL 4] EXPORT, SUMMARIZE, AND BACK UP
import csv
import json
import shutil

filtered_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_web_filtered.csv"
summary_json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_web_summary.json"

# 1. Export the target-class records (header only if there are none)
with open(filtered_csv_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["IP", "Time", "URL", "Status"])
    writer.writerows(target_rows)

# 2. Write the JSON summary
summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "id_sum": ID_SUM,
    "records_processed": records_processed,
    "invalid_records": invalid_records,
    "target_status_class": f"{TARGET_CLASS}xx",
    "records_in_target_class": len(target_rows),
    "count_per_status_code": dict(sorted(status_counts.items())),
    "five_most_requested_urls": [
        {"url": path, "count": count} for path, count in top_urls
    ],
    "note": "" if target_rows else "No records matched the target status class.",
}

with open(summary_json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

# 3. Back up the JSON file and verify the copy
backup_path = BACKUP_DIR / summary_json_path.name
shutil.copy2(summary_json_path, backup_path)

with open(filtered_csv_path, "r", encoding="utf-8") as file:
    exported_rows = sum(1 for _ in file) - 1  # minus the header

print("Filtered CSV:", filtered_csv_path)
print("Exported rows:", exported_rows, "| Expected:", len(target_rows))
print("Row count matches:", exported_rows == len(target_rows))
print("JSON summary:", summary_json_path)
print("Backup:", backup_path)
print("Backup exists:", backup_path.exists())
print("Sizes match:", summary_json_path.stat().st_size == backup_path.stat().st_size)

Filtered CSV: output\Papa_TUPM-26-1927_web_filtered.csv
Exported rows: 0 | Expected: 0
Row count matches: True
JSON summary: output\Papa_TUPM-26-1927_web_summary.json
Backup: backup\Papa_TUPM-26-1927_web_summary.json
Backup exists: True
Sizes match: True
